In [ ]:
from pathlib import Path
import ast
import hashlib
import sys

REPO = Path("/workspace/ProtoSAM-repro")
ARQUIVO = REPO / "models/vmamba.py"

assert ARQUIVO.is_file(), f"Arquivo não encontrado: {ARQUIVO}"

texto = ARQUIVO.read_text()
linhas = texto.splitlines()
arvore = ast.parse(texto)

print("Python:", sys.executable)
print("Arquivo:", ARQUIVO)
print("SHA256:", hashlib.sha256(ARQUIVO.read_bytes()).hexdigest())

def mostrar(no):
    print("\n".join(linhas[no.lineno - 1:no.end_lineno]))

for classe in arvore.body:
    if not isinstance(classe, ast.ClassDef):
        continue

    if classe.name in ("Backbone_VSSM", "LayerNorm2d"):
        print(f"\n===== {classe.name} =====")
        mostrar(classe)

    elif classe.name == "VSSM":
        for metodo in classe.body:
            if isinstance(metodo, ast.FunctionDef) and metodo.name in (
                "forward", "forward_features"
            ):
                print(f"\n===== VSSM.{metodo.name} =====")
                mostrar(metodo)

checkpoint = REPO / (
    "reproducao/eft_tiny_2000_v2_20261011T204725195685Z/"
    "checkpoint_2000.pt"
)
print("\nCheckpoint de referência existe:", checkpoint.is_file())
print(checkpoint)

In [ ]:
from pathlib import Path
import os
import sys
import ast
import importlib

REPO = Path("/workspace/ProtoSAM-repro")
PASTA = REPO / "exp/eft_validacao_polipos_v1"

assert Path(sys.executable).resolve() == Path(
    "/workspace/miniforge3/envs/protosam/bin/python"
).resolve(), f"Kernel incorreto: {sys.executable}"

os.chdir(REPO)

fonte = PASTA / "validar_polipos.py"
assert fonte.is_file(), fonte
codigo = fonte.read_text()

def trocar(antes, depois):
    global codigo
    assert codigo.count(antes) == 1, (
        f"Avaliador diferente do esperado: {antes}"
    )
    codigo = codigo.replace(antes, depois, 1)

trocar(
    "def construir_672(variante):",
    "def construir_672(variante, estagio, norma):",
)

trocar(
    "canais = VARIANTES[variante]['dims'] * 8",
    "canais = VARIANTES[variante]['dims'] * (2 ** estagio)",
)

trocar(
    """            fts = self.encoder(imgs_concat)[-1]
            assert tuple(fts.shape[1:]) == (canais,21,21)""",
    """            capturado = []
            hook = self.encoder.layers[estagio].blocks.register_forward_hook(
                lambda modulo, entradas, saida: capturado.append(saida)
            )
            try:
                self.encoder(imgs_concat)
            finally:
                hook.remove()

            assert len(capturado) == 1 and self.encoder.channel_first
            fts = capturado[0]
            tamanho = 42 if estagio == 2 else 21
            assert tuple(fts.shape[1:]) == (canais,tamanho,tamanho)

            if norma == 'ln_unitaria':
                fts = F.layer_norm(
                    fts.permute(0,2,3,1),
                    (canais,),
                    eps=self.encoder.outnorm3.eps,
                ).permute(0,3,1,2).contiguous()

            assert torch.isfinite(fts).all().item()""",
)

trocar(
    "def executar(piloto, retomar=None):",
    "def executar(piloto, estagio, norma, referencia, retomar=None):",
)

trocar(
    "    assert resumo_piloto['atualizacoes'] == 200",
    "    assert resumo_piloto['atualizacoes'] == 2000 and variante == 'tiny'\n"
    "    assert estagio in (2,3) and norma in ('sem_norm','ln_unitaria')",
)

trocar(
    "ckpt_eft = piloto / 'checkpoint_latest.pt'",
    "ckpt_eft = piloto / 'checkpoint_2000.pt'",
)

trocar(
    "salvo['passo'] == 200 and",
    "salvo['passo'] == 2000 and",
)

trocar(
    "'resolucao_encoder':672, 'canvas':1024, 'estagio':3,",
    "'resolucao_encoder':672, 'canvas':1024, 'estagio':estagio,\n"
    "        'normalizacao_experimento':norma, 'passo_eft':2000,",
)

trocar(
    "'norm_antes':'classifier.norm pré-treinada transferida para outnorm3',",
    "'norm_antes':norma,",
)

trocar(
    "'norm_depois':'outnorm3 do checkpoint EFT',",
    "'norm_depois':norma,",
)

trocar(
    "'finalidade':'Validação anotada antes/depois do piloto; não é teste final',",
    "'finalidade':'Ablação na inferência: camada e normalização; sem novo treinamento',",
)

trocar(
    "    pasta = Path(retomar).resolve() if retomar else "
    "REPO/'reproducao'/f'eft_{variante}_validacao_polipos_{stamp()}'",
    """
    referencia = Path(referencia).resolve()
    ref = ler(referencia / 'config.json')

    for chave in (
        'variante', 'piloto', 'episodios', 'suporte', 'resolucao_encoder',
        'canvas', 'sam', 'selecao_sam', 'alp_inferencia', 'grade_features',
        'kernel_configurado', 'checkpoint_eft_sha256',
        'checkpoint_original_sha256', 'checkpoint_sam_sha256',
        'gpu_driver', 'torch', 'cuda', 'versoes',
        'suporte_tensor_sha256', 'cudnn', 'tf32_matmul', 'tf32_cudnn',
    ):
        assert ref[chave] == config[chave], (
            f'Referência incompatível: {chave}'
        )

    for fonte, digest in config['fontes_sha256'].items():
        if fonte in ref['fontes_sha256']:
            assert ref['fontes_sha256'][fonte] == digest, fonte

    config['referencia'] = str(referencia)
    config['referencia_config_sha256'] = sha(referencia / 'config.json')

    pasta = (
        Path(retomar).resolve()
        if retomar
        else REPO/'reproducao'/f'eft_tiny_camada{estagio}_{norma}_{stamp()}'
    )
""",
)

trocar(
    "modelo = construir_672(variante)",
    "modelo = construir_672(variante, estagio, norma)",
)

trocar(
    "resumo={'variante':variante,'n':145,",
    "resumo={'variante':variante,'estagio':estagio,'norma':norma,"
    "'passo_eft':2000,'n':145,",
)

ast.parse(codigo)
destino = PASTA / "validar_camadas.py"

# Evita sobrescrever silenciosamente outra versão deste experimento.
if destino.exists():
    assert destino.read_text() == codigo, (
        f"Já existe outra versão em {destino}. Envie esta mensagem."
    )
else:
    destino.write_text(codigo)

for caminho in (REPO, PASTA):
    if str(caminho) in sys.path:
        sys.path.remove(str(caminho))
    sys.path.insert(0, str(caminho))

importlib.invalidate_caches()
sys.modules.pop("validar_camadas", None)

from validar_camadas import executar, ler, salvar

print("Avaliador preparado:", destino)
print("Nenhuma avaliação iniciada.")

In [ ]:
import json

TREINO = REPO / (
    "reproducao/eft_tiny_2000_v2_20261011T204725195685Z"
)

CONTROLE_ANTERIOR = REPO / (
    "reproducao/avaliacao_marcos_tiny_20261011T204725195685Z.json"
)
assert CONTROLE_ANTERIOR.is_file(), CONTROLE_ANTERIOR

# Avaliação já concluída da última camada com normalização do checkpoint.
REFERENCIA = Path(
    ler(CONTROLE_ANTERIOR)["avaliacoes"]["2000"]
)
assert (REFERENCIA / "resumo.json").is_file(), REFERENCIA

CONTROLE = REPO / (
    "reproducao/avaliacao_camadas_norm_tiny_2000_v1.json"
)

COMBINACOES = (
    (3, "sem_norm"),
    (3, "ln_unitaria"),
    (2, "sem_norm"),
    (2, "ln_unitaria"),
)

estado = (
    ler(CONTROLE)
    if CONTROLE.exists()
    else {
        "treino": str(TREINO),
        "referencia": str(REFERENCIA),
        "avaliacoes": {},
    }
)
assert estado["treino"] == str(TREINO)
assert estado["referencia"] == str(REFERENCIA)

for estagio, norma in COMBINACOES:
    chave = f"camada{estagio}_{norma}"
    retomar = estado["avaliacoes"].get(chave)

    if retomar is None:
        candidatas = []

        for arquivo in (REPO / "reproducao").glob(
            f"eft_tiny_camada{estagio}_{norma}_*/config.json"
        ):
            cfg = ler(arquivo)
            if (
                cfg.get("piloto") == str(TREINO)
                and cfg.get("referencia") == str(REFERENCIA)
            ):
                candidatas.append(arquivo.parent)

        assert len(candidatas) <= 1, (
            f"Mais de uma execução para {chave}: {candidatas}"
        )
        if candidatas:
            retomar = str(candidatas[0])

    print(f"\n===== {chave} =====", flush=True)

    resultado = executar(
        piloto=TREINO,
        estagio=estagio,
        norma=norma,
        referencia=REFERENCIA,
        retomar=retomar,
    )

    estado["avaliacoes"][chave] = str(resultado)
    salvar(CONTROLE, estado)

comparacao = []

def adicionar(resumo, estagio, norma):
    for condicao, pesos in (
        ("antes", "pre_treinado"),
        ("depois", "EFT_2000"),
    ):
        comparacao.append({
            "pesos": pesos,
            "camada": "ultima" if estagio == 3 else "penultima",
            "normalizacao": norma,
            "Dice_ALP": resumo["ALP"]["Dice"][condicao],
            "Dice_SAM": resumo["SAM"]["Dice"][condicao],
        })

adicionar(
    ler(REFERENCIA / "resumo.json"),
    3,
    "do_checkpoint",
)

for estagio, norma in COMBINACOES:
    chave = f"camada{estagio}_{norma}"
    resumo = ler(
        Path(estado["avaliacoes"][chave]) / "resumo.json"
    )
    adicionar(resumo, estagio, norma)

estado["comparacao"] = comparacao
salvar(CONTROLE, estado)

print("\nCOMPARAÇÃO FINAL — CAMADAS E NORMALIZAÇÃO")
print(json.dumps(comparacao, indent=2, ensure_ascii=False))
print("\nRegistro:", CONTROLE)

In [ ]:
from pathlib import Path
import json
import subprocess

controle = Path(
    "/workspace/ProtoSAM-repro/reproducao/"
    "avaliacao_marcos_tiny_20261011T204725195685Z.json"
)
referencia = Path(
    json.loads(controle.read_text())["avaliacoes"]["2000"]
)
config_anterior = json.loads(
    (referencia / "config.json").read_text()
)

atual = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version",
        "--format=csv,noheader",
    ],
    text=True,
).strip()

print("GPU/driver da referência:", config_anterior["gpu_driver"])
print("GPU/driver atual:        ", atual)

In [ ]:
from pathlib import Path
import ast
import importlib

arquivo = Path(
    "/workspace/ProtoSAM-repro/exp/"
    "eft_validacao_polipos_v1/validar_camadas.py"
)

codigo = arquivo.read_text()
marcador = "# Mudanca de driver registrada explicitamente"

if marcador not in codigo:
    antes = "    ref = ler(referencia / 'config.json')"
    depois = """    ref = ler(referencia / 'config.json')

    # Mudanca de driver registrada explicitamente
    def nomes_gpu(texto):
        return [
            linha.rsplit(',', 1)[0].strip()
            for linha in texto.splitlines()
        ]

    assert nomes_gpu(ref['gpu_driver']) == nomes_gpu(config['gpu_driver']), (
        'Modelo ou quantidade de GPUs diferente da referência.'
    )

    config['ambiente_referencia'] = {
        'gpu_driver': ref['gpu_driver'],
        'driver_diferente': ref['gpu_driver'] != config['gpu_driver'],
    }

    if config['ambiente_referencia']['driver_diferente']:
        print('Driver da referência:', ref['gpu_driver'], flush=True)
        print('Driver desta avaliação:', config['gpu_driver'], flush=True)
"""

    assert codigo.count(antes) == 1
    codigo = codigo.replace(antes, depois, 1)

    trecho = "'gpu_driver', 'torch', 'cuda', 'versoes',"
    assert codigo.count(trecho) == 1
    codigo = codigo.replace(
        trecho,
        "'torch', 'cuda', 'versoes',",
        1,
    )

    ast.parse(codigo)
    arquivo.write_text(codigo)

import validar_camadas
importlib.invalidate_caches()
importlib.reload(validar_camadas)

executar = validar_camadas.executar
ler = validar_camadas.ler
salvar = validar_camadas.salvar

print("Ajuste aplicado. As demais verificações permanecem ativas.")